# Cosmos SAE Training and Feature Browser

Run this notebook on the RunPod checkout. It assumes the repo is at `/workspace/cosmos-sae-reasoner` and that the Cosmos3-Nano Hugging Face cache is under `/workspace/.cache/huggingface`.

The default path builds a robotics-focused manifest from the BridgeData2 synthetic-caption video recipe. Change `DATASET_SAMPLE_COUNT`, `ROBOTICS_RECIPE_WEIGHTS`, `MAX_EXAMPLES`, `TRAIN_STEPS`, and output names in the config cell for larger or cheaper runs.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import time
import sys
import shlex


ROOT = Path('/workspace/cosmos-sae-reasoner')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ENV_FILE = ROOT / '.env'
if ENV_FILE.exists():
    for line in ENV_FILE.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('\"').strip("'"))
PYTHON = shlex.quote(sys.executable)
MODEL_ID = 'nvidia/Cosmos3-Nano'
LAYER = 18

# Robotics-focused dataset. Default to BridgeData2 synthetic captions only.
DATASET_SAMPLE_COUNT = 5000
ROBOTICS_RECIPE_WEIGHTS = {
    'robotics-bridge-captions': 1.0,
}
DATASET_SEED = 0

# Optional: materialize Cosmos RobotSim tar shards to S3 instead of using loose-video repos.
USE_ROBOTSIM_S3 = False
ROBOTSIM_S3_URI = os.environ.get('COSMOS_SAE_ROBOTSIM_S3_URI', '')  # e.g. s3://my-bucket/cosmos/robotsim/run_001
ROBOTSIM_MAX_SHARDS = 1
ROBOTSIM_MAX_SHARD_GB = 1.0

RUN_NAME = f'robotics_l{LAYER}_{DATASET_SAMPLE_COUNT}'
MANIFEST = ROOT / f'outputs/sae_reasoner/manifests/{RUN_NAME}.jsonl'
ACTIVATION_S3_PREFIX = os.environ.get('COSMOS_SAE_ACTIVATION_S3_URI', 's3://cosmos-interpretability/sae_reasoner/activations')
ACTIVATION_DIR = f"{ACTIVATION_S3_PREFIX.rstrip('/')}/{RUN_NAME}" if ACTIVATION_S3_PREFIX else str(ROOT / f'outputs/sae_reasoner/activations/{RUN_NAME}')
SAE_OUT = ROOT / f'outputs/sae_reasoner/saes/{RUN_NAME}.pt'
FEATURES_OUT = ROOT / f'outputs/sae_reasoner/reports/{RUN_NAME}_features.jsonl'
REPORT_OUT = ROOT / f'outputs/sae_reasoner/reports/{RUN_NAME}_features.html'
NEIGHBORS_OUT = ROOT / f'outputs/sae_reasoner/reports/{RUN_NAME}_neighbors.jsonl'
NEIGHBOR_REPORT_OUT = ROOT / f'outputs/sae_reasoner/reports/{RUN_NAME}_neighbors.html'
MANIFEST_S3_URI = f"{ROBOTSIM_S3_URI.rstrip('/')}/manifest.jsonl" if ROBOTSIM_S3_URI else ''

MAX_EXAMPLES = DATASET_SAMPLE_COUNT
COLLECT_PHASE = 'prefill'
MAX_NEW_TOKENS = 128
ACTIVATION_DTYPE = 'bfloat16'  # saved shard dtype; train-sae casts sampled mini-batches to fp32
TRAIN_STEPS = 2000
BATCH_SIZE = 2048
EXPANSION_FACTOR = 16
TOP_K = 32
TOPK_ACTIVATION = 'relu_topk'  # use raw signed 'topk' or 'batch_topk' only for opt-in experiments
BATCH_TOPK_MOMENTUM = 0.01
INIT_METHOD = 'data'
INIT_BLEND = 0.8
ACTIVATION_NORM = 'sqrt_d'
RECON_LOSS = 'mse'
FEATURE_L1_COEFF = 0.0
TRAIN_LR = 3e-4
TRAIN_WARMUP_STEPS = 200
TRAIN_LR_SCHEDULE = 'cosine'
MAX_GRAD_NORM = 1.0  # 0 disables clipping; grad_norm is still logged
TRAIN_SPLITS = 'sae_train'
VAL_SPLITS = 'sae_val'
VAL_BATCH_SIZE = BATCH_SIZE
TRAIN_LOG_EVERY = 10
WANDB_PROJECT = os.environ.get('WANDB_PROJECT', 'cosmos-sae-reasoner')
WANDB_ENTITY = os.environ.get('WANDB_ENTITY', '')
WANDB_MODE = os.environ.get('WANDB_MODE', 'online' if os.environ.get('WANDB_API_KEY') else 'disabled')
WANDB_TAGS = f'cosmos3,sae,layer-{LAYER},all-token,bridge-captions'
USE_WANDB = WANDB_MODE != 'disabled'
FEATURE_IDS = ''  # empty means first 128 features
FEATURE_TOKEN_KINDS = ''  # all by default; use 'video,image,special' for media/special-token reports
FEATURE_PHASES = ''  # all by default; use 'prefill' or 'decode' for control reports
TOP_N = 20

os.environ['HF_HOME'] = '/workspace/.cache/huggingface'
token_path = Path('/root/.cache/huggingface/token')
if token_path.exists() and not os.environ.get('HF_TOKEN'):
    token = token_path.read_text(encoding='utf-8').strip()
    os.environ['HF_TOKEN'] = token
if os.environ.get('HF_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']

def is_s3_uri(value) -> bool:
    return str(value).startswith('s3://')

def split_s3_uri(uri: str):
    rest = uri[len('s3://'):]
    bucket, _, key = rest.partition('/')
    return bucket, key

def s3_client():
    import boto3
    return boto3.client('s3')

def s3_child_uri(uri: str, name: str) -> str:
    return f"{uri.rstrip('/')}/{name.lstrip('/')}"

def list_activation_shards(uri) -> list[str]:
    if is_s3_uri(uri):
        bucket, prefix = split_s3_uri(str(uri))
        prefix = prefix.rstrip('/') + '/' if prefix else ''
        out = []
        paginator = s3_client().get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            for item in page.get('Contents', []):
                key = item.get('Key', '')
                if key.endswith('.pt'):
                    out.append(key.rsplit('/', 1)[-1])
        return sorted(out)
    path = Path(uri)
    return sorted(p.name for p in path.glob('*.pt'))

def read_text_uri(uri) -> str:
    if is_s3_uri(uri):
        bucket, key = split_s3_uri(str(uri))
        return s3_client().get_object(Bucket=bucket, Key=key)['Body'].read().decode('utf-8')
    return Path(uri).read_text(encoding='utf-8')

def uri_exists(uri) -> bool:
    if is_s3_uri(uri):
        bucket, key = split_s3_uri(str(uri))
        try:
            s3_client().head_object(Bucket=bucket, Key=key)
            return True
        except Exception:
            return False
    return Path(uri).exists()

def run(cmd: str) -> None:
    print(f'$ {cmd}', flush=True)
    start = time.time()
    process = subprocess.Popen(cmd, cwd=ROOT, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    rc = process.wait()
    if rc:
        raise subprocess.CalledProcessError(rc, cmd)
    print(f'done in {time.time() - start:.1f}s', flush=True)

print('repo', ROOT)
print('run_name', RUN_NAME)
print('manifest', MANIFEST)
print('activation_dir', ACTIVATION_DIR)
print('sae_out', SAE_OUT)
print('features_out', FEATURES_OUT)
print('report_out', REPORT_OUT)
print('neighbors_out', NEIGHBORS_OUT)
print('neighbor_report_out', NEIGHBOR_REPORT_OUT)
print('use_robotsim_s3', USE_ROBOTSIM_S3)
print('robotsim_s3_uri', ROBOTSIM_S3_URI or '<unset>')
print('robotsim_max_shard_gb', ROBOTSIM_MAX_SHARD_GB)
print('wandb_mode', WANDB_MODE)
print('wandb_project', WANDB_PROJECT if USE_WANDB else '<disabled>')
print('hf_token', '<present>' if os.environ.get('HF_TOKEN') else '<missing>')


## Build Robotics Manifest

Build a configurable robotics manifest. By default this uses streamable loose-video Hugging Face repos. If `USE_ROBOTSIM_S3=True`, it instead extracts a bounded sample from Cosmos RobotSim tar shards, uploads the MP4s to S3, and writes a manifest pointing at those S3 media paths.


In [ ]:
# Build a robotics manifest with a selectable number of samples.
import random


def allocate_counts(total: int, weights: dict[str, float]) -> dict[str, int]:
    if total < 0:
        raise ValueError('total must be non-negative')
    if not weights:
        raise ValueError('weights must not be empty')
    weight_sum = sum(weights.values())
    if weight_sum <= 0:
        raise ValueError('weights must sum to a positive value')
    raw = {name: total * (weight / weight_sum) for name, weight in weights.items()}
    counts = {name: int(value) for name, value in raw.items()}
    remainder = total - sum(counts.values())
    for name, _fraction in sorted(((name, raw[name] - counts[name]) for name in weights), key=lambda item: item[1], reverse=True)[:remainder]:
        counts[name] += 1
    return counts


def build_robotics_manifest(total: int = DATASET_SAMPLE_COUNT, *, recipe_weights: dict[str, float] = ROBOTICS_RECIPE_WEIGHTS, seed: int = DATASET_SEED) -> None:
    counts = allocate_counts(total, recipe_weights)
    tmp_dir = ROOT / 'outputs/sae_reasoner/manifests/_parts'
    tmp_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for offset, (recipe, count) in enumerate(counts.items()):
        if not count:
            continue
        part = tmp_dir / f'{recipe}_{count}.jsonl'
        run(f'{PYTHON} -m tools.sae_reasoner build-corpus-manifest --source recipe --recipe {recipe} --max-records {count} --seed {seed + offset} --output {part}')
        rows.extend(json.loads(line) for line in part.read_text(encoding='utf-8').splitlines())
    random.Random(seed).shuffle(rows)
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    MANIFEST.write_text(''.join(json.dumps(row, ensure_ascii=True, sort_keys=True) + '\n' for row in rows), encoding='utf-8')
    print({'manifest': str(MANIFEST), 'records': len(rows), 'requested': total, 'counts': counts})


def build_robotsim_s3_manifest(total: int = DATASET_SAMPLE_COUNT, *, seed: int = DATASET_SEED) -> None:
    if not ROBOTSIM_S3_URI:
        raise ValueError('Set ROBOTSIM_S3_URI or COSMOS_SAE_ROBOTSIM_S3_URI before enabling USE_ROBOTSIM_S3.')
    env_prefix = 'set -a; [ -f .env ] && . ./.env; set +a; '
    cmd = (
        env_prefix
        + f'{PYTHON} -m tools.sae_reasoner build-corpus-manifest '
        + '--source recipe --recipe physicalai-robotsim '
        + f'--s3-uri {shlex.quote(ROBOTSIM_S3_URI)} '
        + f'--manifest-s3-uri {shlex.quote(MANIFEST_S3_URI)} '
        + f'--max-records {total} --max-shards {ROBOTSIM_MAX_SHARDS} --max-shard-gb {ROBOTSIM_MAX_SHARD_GB} --seed {seed} '
        + f'--output {shlex.quote(str(MANIFEST))}'
    )
    run(cmd)
    print({'manifest': str(MANIFEST), 'manifest_s3_uri': MANIFEST_S3_URI, 'records_requested': total, 'max_shards': ROBOTSIM_MAX_SHARDS, 'max_shard_gb': ROBOTSIM_MAX_SHARD_GB})


BUILD_MANIFEST = True
if BUILD_MANIFEST or not MANIFEST.exists():
    if USE_ROBOTSIM_S3:
        build_robotsim_s3_manifest()
    else:
        build_robotics_manifest()
else:
    print('Using existing manifest', MANIFEST)


## Manifest Browser

Use this before collecting activations to inspect prompts, tags, images, and sampled video frames directly in the notebook. This is intentionally separate from the JSONL file so multimodal rows are easy to scan.

In [ ]:
from html import escape
from IPython.display import HTML, display
import base64
import hashlib
import io
import shutil
import subprocess
import warnings
from urllib.parse import quote
from PIL import Image
from tools.sae_reasoner.manifest import is_remote_media_path, load_manifest
from tools.sae_reasoner.media import materialize_media_path
from tools.sae_reasoner.runtime.cosmos_hf import load_video_frames
try:
    from tqdm.std import TqdmWarning
    warnings.filterwarnings('ignore', category=TqdmWarning, message='IProgress not found.*')
except Exception:
    pass

def _image_data_uri(path: Path, *, max_size=(420, 320)) -> str:
    img = Image.open(path).convert('RGB')
    img.thumbnail(max_size)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')

def _video_contact_sheet_data_uri(path: Path, *, frames=4, thumb=(180, 140), cols=4) -> str:
    arr = load_video_frames(str(path), max_frames=frames)
    thumbs = []
    for i, frame in enumerate(arr):
        img = Image.fromarray(frame)
        img.thumbnail(thumb)
        canvas = Image.new('RGB', (thumb[0], thumb[1] + 22), 'white')
        canvas.paste(img, ((thumb[0] - img.width) // 2, 0))
        from PIL import ImageDraw
        ImageDraw.Draw(canvas).text((6, thumb[1] + 4), f'frame {i}', fill=(0, 0, 0))
        thumbs.append(canvas)
    rows = (len(thumbs) + cols - 1) // cols
    sheet = Image.new('RGB', (cols * thumb[0], rows * (thumb[1] + 22)), 'white')
    for i, img in enumerate(thumbs):
        sheet.paste(img, ((i % cols) * thumb[0], (i // cols) * (thumb[1] + 22)))
    buf = io.BytesIO()
    sheet.save(buf, format='JPEG', quality=85)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')

def _activation_meta_by_id(activation_dir) -> dict:
    metadata_uri = s3_child_uri(activation_dir, 'metadata.jsonl') if is_s3_uri(activation_dir) else Path(activation_dir) / 'metadata.jsonl'
    if not uri_exists(metadata_uri):
        return {}
    rows = [json.loads(line) for line in read_text_uri(metadata_uri).splitlines()]
    return {row.get('id'): row for row in rows}

def _jupyter_file_url(path: Path) -> str:
    try:
        rel = path.resolve().relative_to(ROOT.resolve())
        return '/files/' + '/'.join(quote(part) for part in rel.parts)
    except ValueError:
        return ''

def _video_codec(path: Path) -> str:
    ffprobe = shutil.which('ffprobe')
    if not ffprobe:
        return ''
    result = subprocess.run(
        [ffprobe, '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=codec_name', '-of', 'default=nw=1:nk=1', str(path)],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip().splitlines()[0] if result.returncode == 0 and result.stdout.strip() else ''


def _browser_preview_video_path(path: Path) -> Path | None:
    # Browsers/Jupyter often cannot play AV1-in-MP4 on remote Linux pods.
    # Keep model preprocessing on the original file, but make a small H.264 copy for the HTML player.
    ffmpeg = shutil.which('ffmpeg')
    if not ffmpeg:
        return None
    out_dir = ROOT / 'sae_reasoner_cache' / 'previews'
    out_dir.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256(str(path.resolve()).encode('utf-8')).hexdigest()[:24]
    out = out_dir / f'{digest}.h264.mp4'
    if out.exists() and out.stat().st_mtime >= path.stat().st_mtime and out.stat().st_size > 0:
        return out
    input_args = ['-fflags', '+discardcorrupt', '-err_detect', 'ignore_err', '-hwaccel', 'none']
    if _video_codec(path) == 'av1':
        input_args += ['-c:v', 'libdav1d']
    cmd = [
        ffmpeg, '-y', '-v', 'error', *input_args, '-i', str(path),
        '-map', '0:v:0', '-an',
        '-vf', r'scale=min(640\,iw):-2',
        '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '28',
        '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(out),
    ]
    result = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    if result.returncode != 0 or not out.exists() or out.stat().st_size == 0:
        out.unlink(missing_ok=True)
        return None
    return out


def _local_media_path(media_path: str | None) -> Path | None:
    if not media_path:
        return None
    if is_remote_media_path(media_path):
        return Path(materialize_media_path(media_path, cache_dir=ROOT / '.cache' / 'sae_reasoner' / 'media'))
    path = Path(media_path)
    if not path.is_absolute():
        path = ROOT / path
    return path


def render_manifest_browser(manifest: Path | str, *, activation_dir=None, max_height: int = 760, max_records: int | None = 24, sample_frames: int = 4, make_player: bool = True, show_progress: bool = True) -> None:
    records = load_manifest(manifest)
    total_records = len(records)
    if max_records is not None:
        records = records[:max_records]
    if show_progress:
        print(f'rendering {len(records)} of {total_records} manifest records...', flush=True)
    meta_by_id = _activation_meta_by_id(activation_dir) if activation_dir else {}
    cards = []
    for row_idx, record in enumerate(records, start=1):
        if show_progress:
            print(f'[{row_idx}/{len(records)}] {record.media_type} {record.id}', flush=True)
        media_html = '<div class="missing">text only</div>'
        try:
            media_path = _local_media_path(record.media_path)
            source_path = record.media_path or ''
            if record.media_type == 'image' and media_path:
                media_html = f'<img src="{_image_data_uri(media_path)}" />'
            elif record.media_type == 'video' and media_path:
                preview_path = _browser_preview_video_path(media_path) if make_player else None
                video_url = _jupyter_file_url(preview_path) if preview_path else ''
                player = (
                    f'<video controls preload="metadata" src="{escape(video_url, quote=True)}"></video>'
                    if video_url else '<div class="missing">browser preview unavailable; sampled frames are shown below</div>'
                )
                media_html = (
                    player
                    + f'<details open><summary>sampled frames</summary><img src="{_video_contact_sheet_data_uri(media_path, frames=sample_frames)}" /></details>'
                    + f'<div class="path"><b>source</b> {escape(source_path)}<br/><b>cache</b> {escape(str(media_path))}<br/><b>player</b> {escape(str(preview_path or "unavailable"))}</div>'
                )
        except Exception as exc:
            media_html = f'<div class="missing">preview error: {escape(type(exc).__name__)}: {escape(str(exc))}</div>'
        meta = meta_by_id.get(record.id, {})
        chips = ''.join(f'<span>{escape(tag)}</span>' for tag in record.tags)
        token_bits = ''
        if meta:
            token_bits = (
                f'<div class="tokens"><b>{meta.get("num_tokens", "?")}</b> tokens '
                f'<code>{escape(json.dumps(meta.get("token_kind_counts", {}), sort_keys=True))}</code></div>'
            )
        cards.append(f'''
        <article class="manifest-card">
          <div class="media">{media_html}</div>
          <div class="body">
            <div class="head"><b>{escape(record.id)}</b><span>{escape(record.media_type)}</span></div>
            <p>{escape(record.prompt)}</p>
            <div class="chips">{chips}</div>
            {token_bits}
          </div>
        </article>
        ''')
    display(HTML(f'''
    <style>
      .manifest-scroll {{ max-height: {int(max_height)}px; overflow-y: auto; padding-right: 8px; border: 1px solid #e5e5e5; border-radius: 8px; padding: 10px; background: #fafafa; }}
      .manifest-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(560px, 1fr)); gap: 14px; }}
      .manifest-card {{ display: grid; grid-template-columns: 260px minmax(0, 1fr); gap: 14px; border: 1px solid #ddd; border-radius: 8px; padding: 12px; background: #fff; }}
      .manifest-card img, .manifest-card video {{ max-width: 100%; border-radius: 6px; border: 1px solid #ddd; background: #111; }}
      .manifest-card video {{ width: 100%; display: block; margin-bottom: 8px; }}
      .manifest-card details {{ margin-top: 8px; }}
      .manifest-card summary {{ cursor: pointer; color: #555; font-size: 12px; margin-bottom: 6px; }}
      .manifest-card .head {{ display: flex; justify-content: space-between; gap: 8px; font-size: 15px; }}
      .manifest-card p {{ margin: 10px 0; line-height: 1.35; }}
      .manifest-card .chips {{ display: flex; flex-wrap: wrap; gap: 6px; }}
      .manifest-card .chips span {{ border: 1px solid #ccc; border-radius: 999px; padding: 2px 7px; font-size: 12px; }}
      .manifest-card .tokens, .manifest-card .path, .manifest-card .missing {{ margin-top: 8px; color: #666; font-size: 12px; overflow-wrap: anywhere; }}
      .manifest-card code {{ font-size: 11px; }}
    </style>
    <div class="manifest-scroll"><div class="manifest-grid">{''.join(cards)}</div></div>
    '''))

# Use max_records=None only when you really want to preview every video; it can take a while.
render_manifest_browser(MANIFEST, activation_dir=ACTIVATION_DIR, max_records=24, sample_frames=4, make_player=True)


In [ ]:
# Quick environment check.
run('git rev-parse --short HEAD')
run(f'{PYTHON} -m tools.sae_reasoner inspect-model --model-id nvidia/Cosmos3-Nano --init-mode meta')

In [ ]:
# Optional: collect activations. Defaults to writing activation shards to S3 via ACTIVATION_DIR.
RUN_COLLECTION = False

collect_wandb_args = ''
if USE_WANDB:
    collect_wandb_args = (
        f' --wandb-project {shlex.quote(WANDB_PROJECT)}'
        + (f' --wandb-entity {shlex.quote(WANDB_ENTITY)}' if WANDB_ENTITY else '')
        + f' --wandb-run-name {shlex.quote(RUN_NAME + "_collect")}'
        + f' --wandb-tags {shlex.quote(WANDB_TAGS + ",collection")}'
        + f' --wandb-mode {shlex.quote(WANDB_MODE)}'
    )

if RUN_COLLECTION:
    run(
        f'{PYTHON} -m tools.sae_reasoner collect-activations '
        f'--model-id {MODEL_ID} '
        f'--manifest {MANIFEST} '
        f'--layer {LAYER} '
        f'--output-dir {ACTIVATION_DIR} '
        f'--phase {COLLECT_PHASE} '
        f'--max-new-tokens {MAX_NEW_TOKENS} '
        f'--activation-dtype {ACTIVATION_DTYPE} '
        f'--max-examples {MAX_EXAMPLES}'
        f'{collect_wandb_args}'
    )
else:
    print('Skipping collection. Using existing activation directory/prefix.')

print('activation_uri:', ACTIVATION_DIR)
print('activation shards:', list_activation_shards(ACTIVATION_DIR)[:10])
metadata_uri = s3_child_uri(ACTIVATION_DIR, 'metadata.jsonl') if is_s3_uri(ACTIVATION_DIR) else Path(ACTIVATION_DIR) / 'metadata.jsonl'
print('metadata exists:', uri_exists(metadata_uri))

In [ ]:
# Inspect activation metadata and token-map coverage.
metadata_path = s3_child_uri(ACTIVATION_DIR, 'metadata.jsonl') if is_s3_uri(ACTIVATION_DIR) else Path(ACTIVATION_DIR) / 'metadata.jsonl'
rows = [json.loads(line) for line in read_text_uri(metadata_path).splitlines()] if uri_exists(metadata_path) else []
for row in rows[:10]:
    print({
        'id': row.get('id'),
        'media_type': row.get('media_type'),
        'num_tokens': row.get('num_tokens'),
        'token_kind_counts': row.get('token_kind_counts'),
        'visual_grid': row.get('visual_grid'),
    })

## Nearest Activation Neighbors

Use this after collecting activations to inspect local neighborhoods in activation space before SAE training.

In [ ]:
# Raw activation nearest-neighbor browser. Run after activation collection.
NEIGHBOR_MAX_TOKENS = 5000
NEIGHBOR_NUM_QUERIES = 40
NEIGHBORS_PER_QUERY = 8
NEIGHBOR_QUERY_KINDS = ''  # all token kinds by default; use e.g. 'image,video' or 'text' for control runs

run(
    f'{PYTHON} -m tools.sae_reasoner find-neighbors '
    f'--activation-dir {ACTIVATION_DIR} '
    f'--output {NEIGHBORS_OUT} '
    f'--max-tokens {NEIGHBOR_MAX_TOKENS} '
    f'--num-queries {NEIGHBOR_NUM_QUERIES} '
    f'--neighbors {NEIGHBORS_PER_QUERY} '
    f'--query-kinds "{NEIGHBOR_QUERY_KINDS}" '
    f'--seed {DATASET_SEED}'
)
run(
    f'{PYTHON} -m tools.sae_reasoner render-neighbor-report '
    f'--neighbors {NEIGHBORS_OUT} '
    f'--output {NEIGHBOR_REPORT_OUT}'
)
print('neighbor report:', NEIGHBOR_REPORT_OUT)


In [ ]:
# Preview neighbor rows with token metadata.
neighbor_rows = [json.loads(line) for line in NEIGHBORS_OUT.read_text(encoding='utf-8').splitlines()]
print('num query rows:', len(neighbor_rows))
for row in neighbor_rows[:5]:
    q = row['query']
    qtok = q.get('token_info') or {}
    print('QUERY', {'record_id': q.get('record_id'), 'token_index': q.get('token_index'), 'kind': qtok.get('kind'), 'phase': qtok.get('phase'), 'role': qtok.get('role'), 'token_text': qtok.get('token_text'), 'visual_position': qtok.get('visual_position')})
    for neighbor in row['neighbors'][:3]:
        ntok = neighbor.get('token_info') or {}
        print('  ', {'sim': round(float(neighbor.get('similarity', 0.0)), 4), 'record_id': neighbor.get('record_id'), 'token_index': neighbor.get('token_index'), 'kind': ntok.get('kind'), 'phase': ntok.get('phase'), 'role': ntok.get('role'), 'token_text': ntok.get('token_text'), 'visual_position': ntok.get('visual_position')})


In [ ]:
# Open the nearest-neighbor report inside Jupyter.
from html import escape
from IPython.display import HTML, display

def display_html_report(path, *, height=900):
    html = Path(path).read_text(encoding='utf-8')
    display(HTML(f'<iframe style="width:100%;height:{height}px;border:1px solid #ddd;border-radius:6px;" srcdoc="{escape(html, quote=True)}"></iframe>'))

display_html_report(NEIGHBOR_REPORT_OUT, height=900)


In [ ]:
# Train the SAE. The CLI streams JSON metric rows and optionally logs to W&B.
wandb_args = ''
if USE_WANDB:
    wandb_args = (
        f' --wandb-project {shlex.quote(WANDB_PROJECT)}'
        + (f' --wandb-entity {shlex.quote(WANDB_ENTITY)}' if WANDB_ENTITY else '')
        + f' --wandb-run-name {shlex.quote(RUN_NAME)}'
        + f' --wandb-tags {shlex.quote(WANDB_TAGS)}'
        + f' --wandb-mode {shlex.quote(WANDB_MODE)}'
    )

run(
    f'{PYTHON} -m tools.sae_reasoner train-sae '
    f'--activation-dir {ACTIVATION_DIR} '
    f'--output {SAE_OUT} '
    f'--expansion-factor {EXPANSION_FACTOR} '
    f'--top-k {TOP_K} '
    f'--topk-activation {TOPK_ACTIVATION} '
    f'--batch-topk-momentum {BATCH_TOPK_MOMENTUM} '
    f'--activation-norm {ACTIVATION_NORM} '
    f'--init-method {INIT_METHOD} '
    f'--init-blend {INIT_BLEND} '
    f'--recon-loss {RECON_LOSS} '
    f'--feature-l1-coeff {FEATURE_L1_COEFF} '
    f'--steps {TRAIN_STEPS} '
    f'--batch-size {BATCH_SIZE} '
    f'--lr {TRAIN_LR} '
    f'--warmup-steps {TRAIN_WARMUP_STEPS} '
    f'--lr-schedule {TRAIN_LR_SCHEDULE} '
    f'--max-grad-norm {MAX_GRAD_NORM} '
    f'--train-splits {shlex.quote(TRAIN_SPLITS)} '
    f'--val-splits {shlex.quote(VAL_SPLITS)} '
    f'--val-batch-size {VAL_BATCH_SIZE} '
    f'--log-every {TRAIN_LOG_EVERY}'
    f'{wandb_args}'
)

In [ ]:
# Inspect training metrics.
metrics_path = SAE_OUT.with_suffix('.metrics.jsonl')
metrics = [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines()]
print('num metric rows:', len(metrics))
for row in metrics[-10:]:
    print(row)

try:
    import pandas as pd
    from IPython.display import display
    df = pd.DataFrame(metrics)
    display(df.tail(20))
except Exception as exc:
    print('pandas table unavailable:', exc)

try:
    import matplotlib.pyplot as plt
    xs = [row['step'] for row in metrics]
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    axes[0].plot(xs, [row['loss'] for row in metrics], label='loss')
    axes[0].plot(xs, [row['recon_loss'] for row in metrics], label='recon')
    axes[0].set_title('loss')
    axes[0].legend()
    axes[1].plot(xs, [row.get('explained_variance', 0) for row in metrics])
    if any('val_explained_variance' in row for row in metrics):
        axes[1].plot(xs, [row.get('val_explained_variance', 0) for row in metrics], label='val')
        axes[1].legend()
    axes[1].set_title('explained variance')
    axes[2].plot(xs, [row.get('l0', 0) for row in metrics], label='L0')
    axes[2].plot(xs, [row.get('dead_feature_frac_batch', 0) for row in metrics], label='dead frac')
    axes[2].plot(xs, [row.get('grad_norm', 0) for row in metrics], label='grad norm')
    axes[2].set_title('sparsity')
    axes[2].legend()
    for ax in axes:
        ax.grid(alpha=0.2)
        ax.set_xlabel('step')
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print('metric plot unavailable:', exc)

In [ ]:
# Optional W&B sweep for SAE training dynamics. This is deliberately opt-in.
RUN_SWEEP = False
SWEEP_RUN_CAP = 8
SWEEP_AGENT_COUNT = 8
SWEEP_CONFIG = {
    'method': 'random',
    'metric': {'name': 'val_explained_variance', 'goal': 'maximize'},
    'parameters': {
        'lr': {'values': [1e-4, 3e-4, 1e-3]},
        'batch_size': {'values': [512, 1024, 2048]},
        'top_k': {'values': [16, 32, 64]},
        'expansion_factor': {'values': [8, 16]},
        'warmup_steps': {'values': [100, 200, 500]},
        'lr_schedule': {'values': ['cosine', 'constant']},
        'max_grad_norm': {'values': [1.0, 0.0]},
    },
    'run_cap': SWEEP_RUN_CAP,
}


def run_sweep_command(cmd: str) -> None:
    print(f'$ {cmd}', flush=True)
    process = subprocess.Popen(cmd, cwd=ROOT, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue
        if row.get('event') == 'train_metric':
            metric = {k: v for k, v in row.items() if k != 'event'}
            wandb.log(metric, step=int(metric['step']))
    rc = process.wait()
    if rc:
        raise subprocess.CalledProcessError(rc, cmd)


if RUN_SWEEP:
    import wandb
    if not USE_WANDB:
        raise RuntimeError('Set WANDB_API_KEY or WANDB_MODE before running the sweep.')

    def sweep_train():
        run_ctx = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY or None, tags=WANDB_TAGS.split(',') + ['sweep'])
        cfg = run_ctx.config
        sweep_name = f"{RUN_NAME}_sweep_{run_ctx.id}"
        sweep_out = ROOT / f'outputs/sae_reasoner/saes/{sweep_name}.pt'
        cmd = (
            f'{PYTHON} -m tools.sae_reasoner train-sae '
            f'--activation-dir {ACTIVATION_DIR} '
            f'--output {sweep_out} '
            f'--expansion-factor {int(cfg.expansion_factor)} '
            f'--top-k {int(cfg.top_k)} '
            f'--topk-activation {TOPK_ACTIVATION} '
    f'--batch-topk-momentum {BATCH_TOPK_MOMENTUM} '
    f'--activation-norm {ACTIVATION_NORM} '
            f'--init-method {INIT_METHOD} '
            f'--init-blend {INIT_BLEND} '
            f'--recon-loss {RECON_LOSS} '
            f'--feature-l1-coeff {FEATURE_L1_COEFF} '
            f'--steps {TRAIN_STEPS} '
            f'--batch-size {int(cfg.batch_size)} '
            f'--lr {float(cfg.lr)} '
            f'--warmup-steps {int(cfg.warmup_steps)} '
            f'--lr-schedule {shlex.quote(str(cfg.lr_schedule))} '
            f'--max-grad-norm {float(cfg.max_grad_norm)} '
            f'--train-splits {shlex.quote(TRAIN_SPLITS)} '
            f'--val-splits {shlex.quote(VAL_SPLITS)} '
            f'--val-batch-size {VAL_BATCH_SIZE} '
            f'--log-every {TRAIN_LOG_EVERY} '
            f'--wandb-mode disabled'
        )
        run_sweep_command(cmd)
        metrics_path = sweep_out.with_suffix('.metrics.jsonl')
        if metrics_path.exists():
            final = json.loads(metrics_path.read_text(encoding='utf-8').splitlines()[-1])
            run_ctx.summary.update({f'final_{k}': v for k, v in final.items() if isinstance(v, (int, float))})
            run_ctx.summary['sae_output'] = str(sweep_out)
        run_ctx.finish()

    sweep_id = wandb.sweep(SWEEP_CONFIG, project=WANDB_PROJECT, entity=WANDB_ENTITY or None)
    print('sweep_id:', sweep_id)
    wandb.agent(sweep_id, function=sweep_train, count=SWEEP_AGENT_COUNT, project=WANDB_PROJECT, entity=WANDB_ENTITY or None)
else:
    print('Sweep disabled. Set RUN_SWEEP=True after reviewing SWEEP_RUN_CAP and SWEEP_CONFIG.')


In [ ]:
# Find top activating examples and render the feature browser.
run(
    f'{PYTHON} -m tools.sae_reasoner find-features '
    f'--activation-dir {ACTIVATION_DIR} '
    f'--sae {SAE_OUT} '
    f'--feature-ids "{FEATURE_IDS}" '
    f'--top-n {TOP_N} '
    f'--token-kinds {shlex.quote(FEATURE_TOKEN_KINDS)} '
    f'--phases {shlex.quote(FEATURE_PHASES)} '
    f'--output {FEATURES_OUT}'
)
run(
    f'{PYTHON} -m tools.sae_reasoner render-feature-report '
    f'--features {FEATURES_OUT} '
    f'--output {REPORT_OUT}'
)
print('report:', REPORT_OUT)

In [ ]:
# Preview feature rows with token metadata.
feature_rows = [json.loads(line) for line in FEATURES_OUT.read_text(encoding='utf-8').splitlines()]
print('num feature rows:', len(feature_rows))
for row in feature_rows[:20]:
    token = row.get('token_info') or {}
    print({
        'feature_id': row.get('feature_id'),
        'activation': round(float(row.get('activation', 0.0)), 4),
        'record_id': row.get('record_id'),
        'token_index': row.get('token_index'),
        'kind': token.get('kind'),
        'phase': token.get('phase'),
        'role': token.get('role'),
        'token_text': token.get('token_text'),
        'visual_position': token.get('visual_position'),
    })

In [ ]:
# Open the rendered feature report inside Jupyter.
try:
    display_html_report
except NameError:
    from html import escape
    from IPython.display import HTML, display

    def display_html_report(path, *, height=900):
        html = Path(path).read_text(encoding='utf-8')
        display(HTML(f'<iframe style="width:100%;height:{height}px;border:1px solid #ddd;border-radius:6px;" srcdoc="{escape(html, quote=True)}"></iframe>'))

display_html_report(REPORT_OUT, height=900)